# How to Load an Audio-to-Score Alignment Corpus and Compare Performances

A *Performance Precision* specimen bundles one symbolic score with several
recorded performances of it, plus the audio-to-score alignments that connect
them. This guide loads such a specimen in a single call and arrives at the
question the corpus was built to answer: **how do different pianists shape the
same piece in time?**

The specimen here is Chopin's Nocturne in E♭ major, Op. 9 No. 2 — a `.solo`
score, a Verovio timemap giving each measure's position in quarter notes, and
an `Alignments/` directory holding three alignment files (note, bar, beat) for
each of seven recordings. By the end we will read, for one shared score
position, the seven different moments at which each pianist played it.

The arc:

1. Load the whole specimen in one call.
2. Inspect the measure structure recovered from the timemap.
3. Read the score {{< glossary Timeline >}} (in quarters) and see how a
   `measure+offset` label resolves.
4. Read the per-performer {{< glossary Timeline >}}s (in seconds).
5. Inspect the {{< glossary MatchClaim >}}s, tagged by granularity.
6. Compare the performers' timing through the {{< glossary AlignmentBundle >}}.

## Setup

In [1]:
from __future__ import annotations

from collections import Counter
from fractions import Fraction

import pandas as pd

from timetoalign.core import EnharmonicPitch
from timetoalign.loader.alignment import PerformancePrecisionLoader
from timetoalign.loader.tabular.solo import SoloLoader
from timetoalign.testdata import ensure_data

SPECIMEN_DIR = ensure_data("performance_precision")
SOLO_FILE = SPECIMEN_DIR / "Chopin Nocturne Op. 9 No. 2.solo"

/home/laser/miniconda3/envs/timetoalign/lib/python3.11/site-packages/partitura/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 1. Load the specimen in one call

{{< glossary PerformancePrecisionLoader >}} ingests the whole directory — the
`.solo` score, the Verovio timemap, and every per-performer alignment file —
through the standard two-phase loader pattern. `from_file()` is the one-line
form.

In [2]:
loader = PerformancePrecisionLoader.from_file(SPECIMEN_DIR)
loader

Sources,3 file(s)
,Chopin Nocturne Op. 9 No. 2.solo
,Chopin Nocturne Op. 9 No. 2.json
,Alignments
Events,0
Create,"create_timeline(), create_timelines(), create_bundle()"


Seven recordings, and a few thousand {{< glossary MatchClaim >}}s linking the
score to them. Everything below reads from this single loaded object.

***

## 2. The measure structure

The Verovio timemap records each measure's absolute position in quarter notes.
The loader walks it into a {{< glossary MetricMap >}} — a
{{< glossary ConversionMap >}} from a quarter position to its measure count —
and exposes it directly:

In [3]:
metric_map = loader.metric_map
metric_map

MetricMap(n_measures=38, total_length=425/2)

Thirty-eight measures spanning 212.5 quarters. The companion `.meter` file is
deliberately left unread: it encodes only meter *changes* (four rows across
the whole piece), so it cannot by itself bound the final measure. The
timemap's terminal position supplies that bound, which is why it is preferred
here.

The paired {{< glossary MetricalPositionMap >}} is a
{{< glossary CombinationMap >}} carrying both directions of the
measure↔quarter relationship. `quarters_at(mc, beat)` goes from a metrical
position to a quarter coordinate; `mn_at(quarters)` returns the measure-number
label at a quarter position:

In [4]:
metrical_position_map = loader.metrical_position_map

{
    "downbeat of MC 2 (quarters)": metrical_position_map.quarters_at(2),
    "downbeat of MC 3 (quarters)": metrical_position_map.quarters_at(3),
    "measure label at quarter 0.0": metrical_position_map.mn_at(0.0),
    "measure label at quarter 7.0": metrical_position_map.mn_at(7.0),
}

{'downbeat of MC 2 (quarters)': Fraction(1, 2),
 'downbeat of MC 3 (quarters)': Fraction(13, 2),
 'measure label at quarter 0.0': '1',
 'measure label at quarter 7.0': '3'}

***

## 3. The score timeline

The score is a logical {{< glossary Timeline >}} measured in quarters. It holds
every note of the `.solo` score, each placed at its absolute quarter position.

In [5]:
score_tl = loader.create_timeline("score")
score_tl

ContinuousLogicalTimeline(id='score:clt1', length=425/2, unit=quarters, events=2494, children=0, cmaps=1)

### How a `measure+offset` label becomes a quarter coordinate

Both the `.solo` score and the alignment files write score positions as
`"<measure>+<offset>"`, where the offset is given in **whole notes**. The
resolver is one line of arithmetic — `measure_start + offset × 4` (the `× 4`
converts whole notes to quarters) — with one wrinkle: **measure 0 is the
anacrusis** (the pickup), whose offsets are measured back from a virtual
full-bar downbeat preceding the first sounding note. The very first note of
the piece, labelled `0+11/8`, therefore lands exactly on quarter 0:

In [6]:
first_three = score_tl.get_events().table.slice(0, 3).to_pandas()
first_three[["id", "start"]]

,id,start
0,score:0,"{'value': 0.0, 'numerator': 0, 'denominator': 1}"
1,score:1,"{'value': 0.5, 'numerator': 1, 'denominator': 2}"
2,score:2,"{'value': 0.5, 'numerator': 1, 'denominator': 2}"


A {{< glossary TimeStamp >}} is the primary way to query the score timeline at
a coordinate — here the downbeat of the first full measure, half a quarter in:

In [7]:
score_tl.get_timestamp(Fraction(1, 2))

ID,Coordinate,Type
score:clt1,0.5 quarters,axis


### A note on pitch

The score timeline carries pitch as the raw MIDI pitch integer that the
`.solo` file recorded — faithful to the source, which notes no accidental
spelling. When a *typed* pitch field is wanted, it lives on a freshly composed
`SoloLoader` reading the same `.solo` file. There the field materialises as
`EnharmonicPitch` scalars, and the canonical glyphs ♯/♭ render in their
labels:

In [8]:
solo = SoloLoader.from_file(SOLO_FILE)
pitch_field = solo.events.get_field(EnharmonicPitch)
{"first five pitches": [pitch_field[i] for i in range(5)]}

{'first five pitches': [EnharmonicPitch(B♭4),
  EnharmonicPitch(B♭4),
  EnharmonicPitch(E♭2),
  EnharmonicPitch(G5),
  EnharmonicPitch(E♭2)]}

Because `.solo` records only the MIDI pitch number, an enharmonic pair such as
`G♯3` and `A♭3` both surface as the same number — the field reports what was
represented, not what could be inferred.

***

## 4. The performance timelines

Each recording is a physical {{< glossary Timeline >}} measured in seconds, one
event per aligned note onset. They are retrieved by performer key:

In [9]:
performer_keys = [tl.name for tl in loader.create_timelines() if tl is not score_tl]
performer_keys

['Chopin_Ashkenazy',
 'Chopin_Barenboim',
 'Chopin_Freire',
 'Chopin_Horowitz',
 'Chopin_Pollini',
 'Chopin_Rachmaninoff',
 'Chopin_Rubinstein']

In [10]:
ashkenazy = loader.create_timeline("Chopin_Ashkenazy")
ashkenazy

ContinuousPhysicalTimeline(id='perf:Chopin_Ashkenazy', length=209.142, unit=seconds, events=480, children=0)

A score position is in quarters; a performance position is in seconds. The
alignment is what relates the two, and that is carried by the
{{< glossary MatchClaim >}}s.

***

## 5. The granularity-tagged MatchClaims

The {{< glossary AlignmentBundle >}} assembles the loaded data: the score in
its own group, each performance standalone, and every alignment row as a
cross-group {{< glossary MatchClaim >}}.

In [11]:
bundle = loader.create_bundle()
bundle

AlignmentBundle(id='bundle:AlignmentBundle_1', name='Chopin Nocturne Op. 9 No. 2', timelines=8, groups=1)

Each recording was aligned at three granularities — note, bar, and beat — and
the loader records which one produced each claim in its metadata. Counting the
claims for one performer shows the shape of a single recording's alignment:

In [12]:
ashkenazy_claims = [c for c in bundle.cross_group_claims if c.connects(ashkenazy.id)]
by_granularity = Counter(
    c.metadata.algorithm_params["granularity"] for c in ashkenazy_claims
)
dict(by_granularity)

{'note': 559, 'bar': 32, 'beats': 376}

At the note level, not every score note is found in the recording: where the
aligner located no onset, the row is recorded as a {{< glossary NOMATCH >}}
rather than discarded, so the dangling score position is preserved. Splitting
the note-level claims into located and {{< glossary NOMATCH >}} makes the
distinction explicit:

In [13]:
note_claims = [
    c for c in ashkenazy_claims if c.metadata.algorithm_params["granularity"] == "note"
]

{
    "note claims (total)": len(note_claims),
    "located (synchronous)": sum(1 for c in note_claims if c.is_synchronous),
    "NOMATCH (no onset found)": sum(1 for c in note_claims if not c.is_synchronous),
}

{'note claims (total)': 559,
 'located (synchronous)': 480,
 'NOMATCH (no onset found)': 79}

A single bar-level claim links a score quarter to a performed second. This one
anchors the downbeat of the first full measure (quarter 0.5) to the moment
Ashkenazy played it:

In [14]:
ashkenazy_bar_claims = [
    c for c in ashkenazy_claims if c.metadata.algorithm_params["granularity"] == "bar"
]
ashkenazy_bar_claims[0]

MatchClaim(instant: score:clt1@0.5 <-> perf:Chopin_Ashkenazy@10.3)

***

## 6. Comparing the performances

The corpus exists to compare how performers shape the same music in time. The
{{< glossary AlignmentBundle >}} answers this directly: given one score
coordinate, `get_matchstamp_at` returns the corresponding coordinate on every
connected timeline. A {{< glossary MatchStamp >}} at the first full downbeat is
therefore one score quarter mapped to seven performed seconds:

In [15]:
downbeat = float(metrical_position_map.quarters_at(2))
stamp = bundle.get_matchstamp_at(downbeat, score_tl.id)
stamp

ID,Coordinate,Type
score:clt1,0.5,anchor
perf:Chopin_Rubinstein,1.168,anchor
perf:Chopin_Freire,1.536,anchor
perf:Chopin_Rachmaninoff,5.728,anchor
perf:Chopin_Barenboim,1.2,anchor
perf:Chopin_Pollini,1.952,anchor
perf:Chopin_Horowitz,0.96,anchor
perf:Chopin_Ashkenazy,10.272,anchor


Reading that across a sequence of downbeats gives each performer's arrival time
at each measure — the raw material of a tempo comparison. We take the first
several measure downbeats, ask the bundle for each performer's onset there, and
tabulate the seconds against the score quarters:

In [16]:
downbeat_mcs = range(2, 10)
rows = []
for mc in downbeat_mcs:
    quarters = float(metrical_position_map.quarters_at(mc))
    onset_stamp = bundle.get_matchstamp_at(quarters, score_tl.id)
    row = {"score (quarters)": quarters}
    for key in performer_keys:
        perf_id = loader.create_timeline(key).id
        coord = onset_stamp.coordinates.get(perf_id)
        row[key.replace("Chopin_", "")] = (
            round(float(coord), 3) if coord is not None else None
        )
    rows.append(row)

tempo_table = pd.DataFrame(rows).set_index("score (quarters)")
tempo_table

,Ashkenazy,Barenboim,Freire,Horowitz,Pollini,Rachmaninoff,Rubinstein
score (quarters),,,,,,,
0.5,10.272,1.200,1.536,0.960,1.952,5.728,1.168
6.5,16.152,7.792,8.384,7.232,8.352,13.072,7.696
12.5,21.456,14.304,15.040,14.208,14.848,20.208,14.752
18.5,26.320,21.248,21.408,20.704,20.384,27.520,21.888
24.5,32.672,29.248,28.542,28.352,27.232,35.824,30.032
30.5,38.784,35.920,34.614,35.024,33.664,42.848,36.896
36.5,44.320,42.688,40.998,42.112,40.432,51.696,43.952
42.5,49.600,49.504,47.920,48.976,46.464,60.208,51.344


Each column is one pianist's arrival times in seconds; each row is a shared
score position. The gaps between successive rows within a column are the
inter-downbeat durations — read down a column and the performer's local tempo
is visible, read across a row and the performers' differing placements of the
same beat stand side by side. The same procedure at the beat or note
granularity yields a finer-grained timing profile, all from the one
{{< glossary AlignmentBundle >}}.